# Part 0 — Discovery & Data Profiling (NammaMart)

FDE day-one: profile all six exports, check referential integrity and business rules, and produce the **Data Issues Log**.

Run top to bottom. Outputs:
- `assignment/data_issues_log.csv` — one row per failing record (file, row key, column, problem, dimension, proposed rule, severity)
- A summary by file × dimension, used in `SUBMISSION.md`

All files are read **as text** (`dtype=str`) so nothing is silently coerced — the same way the ELT raw layer will hold them.

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50, "display.width", 200)

# Find the project root (folder containing dags/) whatever the notebook's working directory is
ROOT = Path.cwd()
while not (ROOT / "dags").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "include" / "data" / "nammamart"
OUT_LOG = ROOT / "assignment" / "data_issues_log.csv"
RUN_DATE = pd.Timestamp("2026-10-01")   # the "as of" date for timeliness checks (day after the Sept extract)
print("Project root:", ROOT)
print("Data folder :", DATA, "| files:", sorted(p.name for p in DATA.glob("*.csv")))

Project root: /home/claude/proj
Data folder : /home/claude/proj/include/data/nammamart | files: ['customers.csv', 'orders.csv', 'orders_2026-10-01_increment.csv', 'payments.csv', 'products.csv', 'stores.csv']


## 1. Load every file as raw text

In [2]:
FILES = {
    "orders": "orders.csv",
    "payments": "payments.csv",
    "customers": "customers.csv",
    "products": "products.csv",
    "stores": "stores.csv",
    "orders_increment": "orders_2026-10-01_increment.csv",
}
raw = {k: pd.read_csv(DATA / f, dtype=str, keep_default_na=False) for k, f in FILES.items()}
pd.DataFrame({k: {"rows": len(d), "cols": d.shape[1], "columns": ", ".join(d.columns)} for k, d in raw.items()}).T

,rows,cols,columns
orders,2412,13,"order_id, order_ts, customer_id, store_id, pro..."
payments,2228,7,"payment_id, order_id, payment_ts, payment_mode..."
customers,405,10,"customer_id, full_name, email, phone, city, ar..."
products,152,10,"product_id, product_name, category, sub_catego..."
stores,15,8,"store_id, store_name, zone, area, pincode, ope..."
orders_increment,160,13,"order_id, order_ts, customer_id, store_id, pro..."


## 2. Column profile — blanks, distinct counts, min/max

In [3]:
def num(s):  return pd.to_numeric(s, errors="coerce")
def dt(s):   return pd.to_datetime(s, errors="coerce", format="%Y-%m-%d %H:%M:%S").fillna(
                    pd.to_datetime(s, errors="coerce", format="%Y-%m-%d"))

def profile(name, df):
    rows = []
    for c in df.columns:
        s = df[c]
        blank = (s.str.strip() == "") | s.str.upper().isin(["N/A", "NA", "NULL", "NONE"])
        n, d = num(s), dt(s)
        is_num = n.notna().sum() >= 0.9 * (~blank).sum() and (~blank).sum() > 0
        is_dt = (not is_num) and d.notna().sum() >= 0.9 * (~blank).sum() and (~blank).sum() > 0
        rows.append({
            "file": name, "column": c, "blank_or_NA": int(blank.sum()), "distinct": s.nunique(),
            "type_guess": "numeric" if is_num else "datetime" if is_dt else "text",
            "unparseable": int(((n.isna() if is_num else d.isna() if is_dt else pd.Series(False, index=s.index)) & ~blank).sum()),
            "min": n.min() if is_num else d.min() if is_dt else None,
            "max": n.max() if is_num else d.max() if is_dt else None,
        })
    return pd.DataFrame(rows)

col_profile = pd.concat([profile(k, d) for k, d in raw.items()], ignore_index=True)
col_profile

,file,column,blank_or_NA,distinct,type_guess,unparseable,min,max
0,orders,order_id,0,2400,text,0,None,None
1,orders,order_ts,2,2393,datetime,4,2026-09-01 06:30:15,2027-01-01 09:00:00
2,orders,customer_id,18,401,text,0,None,None
3,orders,store_id,0,18,text,0,None,None
4,orders,product_id,0,159,text,0,None,None
...,...,...,...,...,...,...,...,...
56,orders_increment,order_amount,0,148,numeric,0,11.0,2052.0
57,orders_increment,payment_mode,0,4,text,0,None,None
58,orders_increment,order_status,0,2,text,0,None,None
59,orders_increment,channel,0,1,text,0,None,None


## 3. Duplicate keys

In [4]:
KEYS = {"orders": "order_id", "payments": "payment_id", "customers": "customer_id",
        "products": "product_id", "stores": "store_id", "orders_increment": "order_id"}
pd.DataFrame([{
    "file": k, "key": KEYS[k],
    "duplicate_keys": int(d[KEYS[k]].duplicated().sum()),
    "fully_identical_rows": int(d.duplicated().sum()),
} for k, d in raw.items()])

,file,key,duplicate_keys,fully_identical_rows
0,orders,order_id,12,12
1,payments,payment_id,4,4
2,customers,customer_id,5,0
3,products,product_id,2,0
4,stores,store_id,0,0
5,orders_increment,order_id,0,0


## 4. Distinct values of category columns

In [5]:
CATS = {"orders": ["order_status", "payment_mode", "channel"],
        "payments": ["payment_mode", "payment_status"],
        "customers": ["loyalty_tier", "city", "is_active"],
        "products": ["category", "unit", "is_perishable"],
        "stores": ["zone"]}
for f, cols in CATS.items():
    for c in cols:
        print(f"{f}.{c}: {raw[f][c].map(repr).value_counts().to_dict()}")

orders.order_status: {"'DELIVERED'": 2182, "'CANCELLED'": 156, "'RETURNED'": 69, "'Pending??'": 3, "'delivered'": 2}
orders.payment_mode: {"'UPI'": 1329, "'Card'": 493, "'Cash on Delivery'": 326, "'Wallet'": 254, "'upi'": 4, "'Cashh'": 2, "'UPI '": 2, "'CARD'": 1, "'cod'": 1}
orders.channel: {"'App'": 2044, "'Web'": 368}
payments.payment_mode: {"'UPI'": 1238, "'Card'": 445, "'Cash on Delivery'": 301, "'Wallet'": 234, "'upi'": 4, "'UPI '": 2, "'Cashh'": 2, "'cod'": 1, "'CARD'": 1}
payments.payment_status: {"'SUCCESS'": 2116, "'REFUNDED'": 68, "'FAILED'": 44}
customers.loyalty_tier: {"'Bronze'": 185, "'Silver'": 133, "'Gold'": 54, "'Platinum'": 25, "'BRONZE'": 3, "'bronze'": 3, "'SILVER'": 2}
customers.city: {"'Bengaluru'": 399, "''": 6}
customers.is_active: {"'Y'": 352, "'N'": 53}
products.category: {"'Dairy & Bakery'": 26, "'Household'": 26, "'Fruits & Vegetables'": 25, "'Snacks & Beverages'": 25, "'Personal Care'": 25, "'Staples'": 22, "''": 3}
products.unit: {"'pack'": 76, "'unit'": 

## 5. Rule checks → Data Issues Log

All rules live in **`include/dq_rules.py`** — the same module the ETL DAG imports, and the spec the ELT SQL mirrors.
Severity: **Critical** → quarantine; **Warning** → load and log. Agreed decisions D1–D3 are documented at the top of that module.

In [6]:
import sys, importlib
sys.path.insert(0, str(ROOT))
import include.dq_rules as dq
importlib.reload(dq)

# _source_file tells the timeliness rules which period each file covers (Sept export vs 1 Oct increment)
master_raw = {k: raw[k].assign(_source_file=FILES[k]) for k in ["orders", "payments", "customers", "products", "stores"]}
ctx = dq.build_context(master_raw, RUN_DATE)

validated, dq_results, issue_frames = {}, [], []
for name, df in master_raw.items():
    passed, failed, results, issues = dq.validate(name, df, ctx, FILES[name])
    validated[name] = (passed, failed)
    dq_results.append(results.assign(file=FILES[name]))
    issue_frames.append(issues)

# Increment file: same order rules; its order_ids are known orders for FK purposes
ctx_inc = {**ctx, "order_ids": ctx["order_ids"] | set(raw["orders_increment"].order_id)}
p_inc, f_inc, r_inc, i_inc = dq.validate("orders", raw["orders_increment"].assign(_source_file=FILES["orders_increment"]),
                                          ctx_inc, FILES["orders_increment"])
dq_results.append(r_inc.assign(file=FILES["orders_increment"])); issue_frames.append(i_inc)

dq_results = pd.concat(dq_results, ignore_index=True)
issues = pd.concat(issue_frames, ignore_index=True)
issues.to_csv(OUT_LOG, index=False)

print(f"{len(issues)} issue rows → {OUT_LOG.relative_to(ROOT)}\n")
pd.DataFrame({FILES[k]: {"rows": len(raw[k]), "passed": len(p), "quarantined": len(f)}
              for k, (p, f) in validated.items()} | {FILES["orders_increment"]: {"rows": len(raw["orders_increment"]),
              "passed": len(p_inc), "quarantined": len(f_inc)}}).T

209 issue rows → assignment/data_issues_log.csv



,rows,passed,quarantined
orders.csv,2412,2324,88
payments.csv,2228,2199,29
customers.csv,405,397,8
products.csv,152,142,10
stores.csv,15,14,1
orders_2026-10-01_increment.csv,160,160,0


### Scorecard — every check (this is what `dq_results` will hold each run)

In [7]:
dq_results[["file", "check_name", "dimension", "severity", "rows_checked", "rows_failed", "status"]]

,file,check_name,dimension,severity,rows_checked,rows_failed,status
0,orders.csv,orders_order_id_unique,Uniqueness,Critical,2412,12,FAIL
1,orders.csv,orders_customer_id_not_null,Completeness,Critical,2412,18,FAIL
2,orders.csv,orders_order_ts_valid,Validity,Critical,2412,6,FAIL
3,orders.csv,orders_order_ts_not_future,Timeliness,Critical,2412,7,FAIL
4,orders.csv,orders_status_casing,Consistency,Warning,2412,2,WARN
5,orders.csv,orders_status_allowed,Validity,Critical,2412,3,FAIL
6,orders.csv,orders_payment_mode_casing,Consistency,Warning,2412,10,WARN
7,orders.csv,orders_payment_mode_allowed,Validity,Critical,2412,0,PASS
8,orders.csv,orders_quantity_positive,Accuracy,Critical,2412,10,FAIL
9,orders.csv,orders_discount_range,Validity,Critical,2412,3,FAIL


### Issues by file × dimension

In [8]:
pd.crosstab(issues.file, issues.dimension, margins=True)

dimension,Accuracy,Completeness,Consistency,Timeliness,Uniqueness,Validity,All
file,,,,,,,
customers.csv,0,6,8,3,5,22,44
orders.csv,31,26,26,7,12,12,114
payments.csv,10,0,18,7,4,0,39
products.csv,5,3,0,0,2,0,10
stores.csv,1,1,0,0,0,0,2
All,47,36,52,17,23,34,209


In [9]:
(issues.groupby(["file", "check_name", "dimension", "severity", "root_cause"])
       .agg(rows=("row_key", "size"), example_keys=("row_key", lambda s: ", ".join(s.unique()[:4])))
       .reset_index().sort_values(["file", "severity", "rows"], ascending=[True, True, False]))

,file,check_name,dimension,severity,root_cause,rows,example_keys
1,customers.csv,customers_customer_id_unique,Uniqueness,Critical,Customer moved / changed number -> candidate f...,5,"C00103, C00252, C00226, C00036"
5,customers.csv,customers_signup_not_future,Timeliness,Critical,Bad date entry,3,"C00400, C00398, C00038"
2,customers.csv,customers_email_valid,Validity,Warning,No validation at signup,12,"C00337, C00307, C00063, C00045"
6,customers.csv,customers_tier_casing,Consistency,Warning,Free-text entry,8,"C00199, C00171, C00133, C00032"
3,customers.csv,customers_phone_valid,Validity,Warning,Digits dropped at entry,7,"C00151, C00240, C00095, C00357"
0,customers.csv,customers_city_present,Completeness,Warning,Optional field at signup,6,"C00355, C00394, C00233, C00142"
4,customers.csv,customers_pincode_valid,Validity,Warning,Partial pincode (5600XX),3,"C00132, C00083, C00166"
8,orders.csv,orders_customer_id_not_null,Completeness,Critical,Order captured without customer link,18,"ORD002255, ORD000701, ORD000997, ORD000580"
7,orders.csv,orders_amount_formula,Accuracy,Critical,Checkout pricing bug: amount is 10x / 1.5x / 0...,15,"ORD002305, ORD002004, ORD000845, ORD000964"
12,orders.csv,orders_order_id_unique,Uniqueness,Critical,Export duplicated rows,12,"ORD001462, ORD001441, ORD001692, ORD002129"


## 6. Revenue impact of the quarantine

In [10]:
def delivered_revenue(df):
    st = dq.std_status(df.order_status)
    return dq.num(df.order_amount)[st == "DELIVERED"].sum()

orders_dedup = raw["orders"].drop_duplicates("order_id")
o_pass, o_fail = validated["orders"]
pd.Series({
    "Delivered revenue — raw (deduped)": delivered_revenue(orders_dedup),
    "Delivered revenue — after Critical checks": delivered_revenue(o_pass),
}).round(2).to_frame("₹")

,₹
Delivered revenue — raw (deduped),1210776.78
Delivered revenue — after Critical checks,1142886.50


## 7. Reconciliation — orders ↔ payments

Classification agreed with the client (D1, D2). Every non-cancelled order and every unmatched payment gets a `recon_status`.
`variance` = cash collected − cash we should have collected (negative = money not received, positive = over-collected).

In [11]:
recon = dq.reconcile(raw["orders"], o_fail, raw["payments"])
summary = (recon.groupby(["recon_status", "owner"])
                .agg(rows=("recon_status", "size"), variance=("variance", "sum"))
                .round(2).reset_index().sort_values("rows", ascending=False))
summary

,recon_status,owner,rows,variance
3,MATCHED,-,2019,0.00
9,REFUND_MATCHED,-,67,0.00
6,PAYMENT_FOR_QUARANTINED_ORDER,Data team,50,518.90
1,DELIVERED_PAYMENT_FAILED,Collections,42,-25722.25
0,DELIVERED_NO_PAYMENT,Finance + Tech (payment export),27,-16664.35
7,PRICING_ERROR_AT_CHECKOUT,Tech (checkout) + Finance,15,36480.33
8,PROMO_SHORTFALL,Marketing / promo ops,14,-785.00
4,NEGATIVE_ORDER,Ops (returns keyed as orders),10,0.00
5,ORPHAN_PAYMENT,Payments team,8,4580.57
2,DUPLICATE_PAYMENT,Payments team,4,1330.50


In [12]:
# D2: delivered orders that are excluded from net revenue until payment is confirmed
held = recon[recon.recon_status.isin(dq.REVENUE_BLOCKING_RECON)]
print(f"Delivered orders held out of net revenue: {len(held)}  (₹{held.expected_amount.sum():,.2f})")
recon[recon.recon_status == "PRICING_ERROR_AT_CHECKOUT"][["order_id", "formula_amount", "expected_amount", "received_amount", "variance"]]

Delivered orders held out of net revenue: 69  (₹42,386.60)


,order_id,formula_amount,expected_amount,received_amount,variance
290,ORD002305,675.75,6757.50,6757.50,6081.75
446,ORD002004,672.00,6720.00,6720.00,6048.00
714,ORD000845,471.20,4712.00,4712.00,4240.80
749,ORD000964,837.00,8370.00,8370.00,7533.00
953,ORD002298,610.20,305.10,305.10,-305.10
955,ORD002031,321.00,3210.00,3210.00,2889.00
1196,ORD000801,3645.00,5467.50,5467.50,1822.50
1339,ORD000880,345.10,517.65,517.65,172.55
1531,ORD001274,220.00,110.00,110.00,-110.00
1534,ORD000677,262.80,394.20,394.20,131.40


## 8. Increment file — what will the MERGE do?

In [13]:
inc, o = raw["orders_increment"], raw["orders"]
existing = inc.order_id.isin(o.order_id)
print(f"Increment rows: {len(inc)} | new order_ids: {(~existing).sum()} | corrections to existing orders: {existing.sum()}")
inc[existing].merge(o.drop_duplicates("order_id")[["order_id", "order_status"]], on="order_id",
                    suffixes=("_new", "_old"))[["order_id", "order_status_old", "order_status_new", "order_ts"]]

Increment rows: 160 | new order_ids: 150 | corrections to existing orders: 10


,order_id,order_status_old,order_status_new,order_ts
0,ORD000017,DELIVERED,RETURNED,2026-09-01 09:38:13
1,ORD000271,DELIVERED,RETURNED,2026-09-04 20:42:53
2,ORD000332,DELIVERED,RETURNED,2026-09-05 16:48:59
3,ORD000337,DELIVERED,RETURNED,2026-09-05 17:54:32
4,ORD000584,DELIVERED,RETURNED,2026-09-08 20:17:36
5,ORD000708,DELIVERED,RETURNED,2026-09-10 12:44:47
6,ORD000822,DELIVERED,RETURNED,2026-09-11 22:25:24
7,ORD001025,DELIVERED,RETURNED,2026-09-14 10:26:30
8,ORD001067,DELIVERED,RETURNED,2026-09-14 19:29:01
9,ORD001990,DELIVERED,RETURNED,2026-09-25 20:53:03


## 9. Questions to ask the client before building

1. **Timestamps:** `order_ts` has values like `31/09/2026`, `2026-13-05` and `N/A`. Is `order_ts` the order-placed time from the app server, and can we get a corrected export — or should these rows stay quarantined until ops fixes them at source?
2. **Revenue definition vs payments:** finance reconciles to the gateway. For a `SUCCESS` payment that's lower than `order_amount` (e.g. ₹50/₹100 short), which number is "truth" for the CFO — order revenue or cash collected? And is a RETURNED order's revenue reversed in the month of the return or the month of the original sale?
3. **Duplicates in masters:** `C00036` appears twice with different phones/areas, and `P0030` has a current and an "(old)" row with different MRPs. Which record wins — latest, or does the "(old)" MRP apply to orders placed before a change date (i.e. do you need price history)?
4. *(Optional)* **Delivery outliers:** 480 / 900 / 1440-minute deliveries — data-entry (hours vs minutes) or genuine failed deliveries that should be excluded from the 10-minute-promise KPI but kept in revenue?